# Install the autorift environment with Pixi

## Install the environment and register the kernel with ipykernel

In [ ]:
!pixi install -e autorift

env_name = "autorift"
display_name = f'"{env_name} (Python)"'

!pixi run -e autorift python -m ipykernel install \
  --user \
  --name $env_name \
  --display-name $display_name

## Ensure notebook shell commands run in the autorift environment
This ensures that shell commands executed from inside a notebook with ! run in the notebook kernel’s environment. This works by launching the entire Jupyter kernel process inside the Pixi environment, so the kernel’s PATH, environment variables, and Python executable all come from that Pixi environment, not from the parent JupyterLab environment.

In [ ]:
from pathlib import Path
from jupyter_client.kernelspec import KernelSpecManager
import json

ksm = KernelSpecManager()
spec = ksm.get_kernel_spec(env_name)
kernel_dir = Path(spec.resource_dir)
kernel_json = kernel_dir / "kernel.json"

data = json.loads(kernel_json.read_text())
orig_argv = data.get("argv", [])

new_argv = [
    "pixi",
    "run",
    "--manifest-path",
    str(Path.cwd()),
    "-e",
    env_name,
] + orig_argv

data["argv"] = new_argv
kernel_json.write_text(json.dumps(data, indent=2))
print(f"Updated kernel.json at {kernel_json} with Pixi wrapper.")
print("argv:", data["argv"])